# Ansible Assignment

**Task:** Install and configure Ansible, Puppet, or Chef on a local or cloud environment.
Write and execute basic configuration scripts.

We'll use **Ansible** (agentless, easiest to set up for this exercise).

## 1. Installation
```bash
# Control node (your machine)
sudo apt update
sudo apt install -y software-properties-common
sudo add-apt-repository --yes --update ppa:ansible/ansible
sudo apt install -y ansible
ansible --version
```
For managed nodes, only Python + SSH access are required — no agent to install.

## 2. Inventory File

In [1]:
import os
os.makedirs('project', exist_ok=True)

with open('project/inventory.ini', 'w') as f:
    f.write('''[webservers]
web1 ansible_host=192.168.56.10 ansible_user=ubuntu
web2 ansible_host=192.168.56.11 ansible_user=ubuntu

[dbservers]
db1 ansible_host=192.168.56.20 ansible_user=ubuntu

[all:vars]
ansible_ssh_private_key_file=~/.ssh/id_rsa
''')
print("Created project/inventory.ini")
print("Test connectivity: ansible all -i project/inventory.ini -m ping")


Created project/inventory.ini
Test connectivity: ansible all -i project/inventory.ini -m ping


## 3. Basic Configuration Playbook
A simple playbook that updates packages, installs common tools, and creates a user — demonstrating core Ansible modules (`apt`, `user`, `copy`, `service`).

In [2]:
with open('project/basic_setup.yml', 'w') as f:
    f.write('''---
- name: Basic server configuration
  hosts: all
  become: true

  vars:
    app_user: deploy

  tasks:
    - name: Update apt cache
      apt:
        update_cache: yes
        cache_valid_time: 3600

    - name: Install common packages
      apt:
        name:
          - curl
          - git
          - vim
          - htop
        state: present

    - name: Ensure deploy user exists
      user:
        name: "{{ app_user }}"
        shell: /bin/bash
        create_home: yes

    - name: Copy a sample config file
      copy:
        content: "managed_by=ansible\nenv=staging\n"
        dest: /etc/app.conf
        owner: "{{ app_user }}"
        mode: "0644"

    - name: Ensure a marker file shows the playbook ran
      file:
        path: /etc/ansible_provisioned
        state: touch
''')
print("Created project/basic_setup.yml")
print("Run: ansible-playbook -i project/inventory.ini project/basic_setup.yml")


Created project/basic_setup.yml
Run: ansible-playbook -i project/inventory.ini project/basic_setup.yml


## 4. Ad-hoc Commands (quick one-off checks)
```bash
ansible all -i project/inventory.ini -m ping
ansible webservers -i project/inventory.ini -a "uptime"
ansible all -i project/inventory.ini -m apt -a "name=curl state=present" --become
```